### DistilBERT and CrisisMMD: Calibration and Privacy Budget Experiments

This report evaluates DistilBERT as a model for adaptive differential privacy (DPA-DP) in a simulated disaster scenario, following the earlier calibration work on MobileNetV3.

Without privacy noise, DistilBERT was calibrated on the Emotion dataset (learning rate 0.0001, at least one unfrozen transformer layer, 87-90% accuracy) and then on the text part of CrisisMMD 2.0 (67-69% accuracy, a lower ceiling because the tweets are noisier and the labels more ambiguous).

With DP-SGD enabled, a clipping norm sweep showed that accuracy is unchanged from 0.0001 to 1 and drops only below that, so 0.0001 was used. Four noise-multiplier profiles (one fixed and three DPA-DP) were then compared on a simulated crisis: 180 normal rounds, then 20 rounds with a mix of Emotion and CrisisMMD messages, all with a matched final privacy budget of about epsilon 14.2.

All DPA-DP profiles were slightly worse than the fixed baseline before the crisis but better after it. The profile with the largest phase difference (1.35 normal, 0.65 disaster) performed best, gaining about 1.6 points overall and 4.4 points on disaster messages over the fixed baseline.


##### page break

In [ ]:
from matplotlib import pyplot
import opacus.accountants
from opacus.accountants import RDPAccountant, PRVAccountant

delta = 1e-5

def epsilon(noise_multiplier_1, noise_multiplier_2, accountant, m=1):
    filename = f"{type(accountant).__name__}_{noise_multiplier_1:.2f}_{noise_multiplier_2:.2f}_{noise_multiplier_switch}_{rounds}_{batch_size}_{n_samples}_{m}.txt"
    try:
        with open(filename) as file:
            return [float(line) for line in file]
    except FileNotFoundError:
        pass
    n_batches = (n_samples * m + batch_size - 1) // batch_size
    sample_rate = m / n_batches
    epsilons = [0]
    for round_idx in range(rounds):
        noise_multiplier = noise_multiplier_1 if round_idx < noise_multiplier_switch else noise_multiplier_2
        for _ in range(n_batches):
            accountant.step(noise_multiplier=noise_multiplier, sample_rate=sample_rate)
        epsilons.append(float(accountant.get_epsilon(delta=delta)))
    # eps, best_alpha = accountant.get_privacy_spent(delta=delta)
    # print(best_alpha)
    with open(filename, "w") as file:
        for value in epsilons:
            file.write(f"{value}\n")
    return epsilons

def plot_epsilon(data, ylabel="epsilon"):
    figure = pyplot.figure(figsize=(13,4))
    for name, arr in data.items():
        pyplot.plot(arr, label=name)
    pyplot.xlabel("round")
    pyplot.ylabel(ylabel)
    pyplot.grid(True, alpha=0.3)
    pyplot.legend()
    pyplot.show()
    # figure.savefig("epsilon.pdf", bbox_inches="tight")

In [ ]:
from matplotlib import pyplot
import pandas
def flwr_accuracy_sub(ax, file, name, start=0, key="accuracy", loc=None):
    df = pandas.read_csv(file + ".csv")
    for id_val, g in df.groupby("id", sort=False):
        g = g.sort_values("round").tail(len(g) - start)
        ax.plot(g["round"], g[key], label=str(id_val))

    ax.set_xlabel("round")
    ax.set_ylabel(name)
    ax.grid(True, alpha=0.3)
    if loc is not None:
        ax.legend(loc=loc)

In [ ]:
import pandas
from matplotlib import pyplot
def fixed_diff(data):
    fixed = (data[data["id"] == "fixed"][["seed", "round", "accuracy", "loss"]]
        .rename(columns={"accuracy": "fixed_accuracy", "loss": "fixed_loss",}))
    data_diff = data.merge(fixed, on=["seed", "round"])
    data_diff["accuracy"] = data_diff["accuracy"] - data_diff["fixed_accuracy"]
    data_diff["loss"] = data_diff["loss"] - data_diff["fixed_loss"]
    data_diff = data_diff.drop(columns=["fixed_accuracy", "fixed_loss"])
    return data_diff
def flwr_accuracy_error(ax, file, start=0, key="accuracy"):
    data = pandas.read_csv(file + ".csv")
    data = data[data["round"] >= start]
    data = fixed_diff(data)
    summary = (data.groupby(["id", "round"], sort=False)[key].agg(mean="mean", std="std").reset_index())
    pyplot.figure(figsize=(11, 6))
    for id_val, group in summary.groupby("id", sort=False):
        group = group.sort_values("round")
        if id_val == "fixed":
            ax.plot(group["round"], group["mean"], label=str(id_val))
        else:
            ax.errorbar(group["round"], group["mean"], yerr=group["std"], linewidth=2, capsize=4, alpha=0.7, label=str(id_val))
    ax.set_xlabel("round")
    ax.set_ylabel(key + " diff")
    ax.grid(True, alpha=0.3)
    ax.legend()
def flwr_accuracy(ax, file, start=0, key="accuracy"):
    data = pandas.read_csv(file + ".csv")
    data = data[data["round"] >= start]
    summary = (data.groupby(["id", "round"], sort=False)[key].agg(mean="mean", std="std").reset_index())
    for id_val, group in summary.groupby("id", sort=False):
        group = group.sort_values("round")
        line, = ax.plot(group["round"], group["mean"], label=str(id_val))
        ax.fill_between(
        group["round"],
        group["mean"] - group["std"],
        group["mean"] + group["std"],
        color=line.get_color(),
        alpha=0.2
    )

    ax.set_xlabel("round")
    ax.set_ylabel(key)
    ax.grid(True, alpha=0.3)
    ax.legend()

In [ ]:
def accuracy_summary(file):
    data = pandas.read_csv(file + ".csv")
    data = data[data["round"] > 190]
    fixed = (
        data[data["id"] == "fixed"]
        [["seed", "round", "accuracy"]]
        .rename(columns={"accuracy": "fixed"})
    )
    data = data.merge(fixed, on=["seed", "round"], how="left")
    data["fixed"] = data["accuracy"] - data["fixed"]
    summary = (data.groupby("id", sort=False)["accuracy"].agg(mean="mean", deviation="std").reset_index())
    summary_diff = (data.groupby("id", sort=False)["fixed"].agg(diff="mean", dev_diff="std").reset_index())
    summary = summary.merge(summary_diff, on=["id"], how="left")
    summary = summary.set_index("id")
    summary.index.name = None
    summary = summary.style.format("{:.4f}")
    return summary

##### page break

### DistilBERT performance by learning rate (August)
noise_multiplier: 0, tune_layers: 1, batch_size: 32

DistilBERT model on the Emotion dataset (16,000 training / 2,000 test texts) with privacy noise disabled. Accuracy and loss both improved steadily as the learning rate increased from 1e-7 to 0.001, but at 0.01 and above the model stopped learning, producing effectively random output. Learning rate 0.0001 was selected for the following experiments as it gives good results and far enough from the collapse value.

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(13, 4))
flwr_accuracy_sub(ax1, "../05_CrisisMMD/bert_clear_learning", "accuracy")
flwr_accuracy_sub(ax2, "../05_CrisisMMD/bert_clear_learning", "loss", key="loss", loc="upper right")

##### page break

### DistilBERT performance by frozen layers (August)
noise_multiplier: 0, learning_rate: 0.0001, batch_size: 32

With layers 2, 3 and above unfrozen, the model converges to 88-90% accuracy on the Emotion dataset. However, a fully frozen backbone with only the classifier head trainable, was unable to provide result better than 60%, indicating that at least one unfrozen backbone layer is necessary for DistilBERT performance.

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(13, 4))
flwr_accuracy_sub(ax1, "../05_CrisisMMD/bert_clear_layers", "accuracy")
flwr_accuracy_sub(ax2, "../05_CrisisMMD/bert_clear_layers", "loss", key="loss", loc="upper right")

##### page break

### DistilBERT performance on CrisisMMD (August)
noise_multiplier: 0, learning_rate: 0.0001, batch_size: 32

In this test the Emotion dataset was replaced with the text modality of CrisisMMD 2.0, a state of the art dataset of social media messages and images from various disaster types. It defines 8 labels and is pre-split by its authors into 13,608 training and 2,237 test texts.

Configurations with layer 1 and above unfrozen converge to 67-69% accuracy. Models with more unfrozen layers show a higher rise in loss value during later training rounds.

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(13, 4))
flwr_accuracy_sub(ax1, "../05_CrisisMMD/crisismmd_layers", "accuracy")
flwr_accuracy_sub(ax2, "../05_CrisisMMD/crisismmd_layers", "loss", key="loss", loc="upper right")

##### page break

### DistilBERT DPA-DP privacy budget allocation
Epsilon budget consumption over 200 rounds (180 normal + 20 simulated disaster phase), for four noise-multiplier profiles tuned to reach a matched final budget of PRV epsilon around 14.2.
- fixed - uniform noise multiplier 1.0 across both phases
- dpa-dp-0.85 - low profile: 1.03 normal, 0.85 disaster
- dpa-dp-0.75 - medium profile: 1.08 normal, 0.75 disaster
- dpa-dp-0.65 - high profile: 1.35 normal, 0.65 disaster, the largest phase difference

In [ ]:
rounds=200
noise_multiplier_switch = 180
batch_size = 32
n_samples = 1600 # emotion 16000 training set / 10 clients

data = {}
data["fixed"] = epsilon(1.00, 1.00, PRVAccountant())
data["dpa-dp-0.85"] = epsilon(1.03, 0.85, PRVAccountant())
data["dpa-dp-0.75"] = epsilon(1.08, 0.75, PRVAccountant())
data["dpa-dp-0.65"] = epsilon(1.35, 0.65, PRVAccountant())
print({k: a[-1] for k, a in data.items()})
plot_epsilon(data, "PRV epsilon")

##### page break

### Text dataset for crisis simulation
Similarly to the MobileNetV3 experiment, we created a dataset containing normal messages for the normal stage (16,000 texts from the Emotion dataset). After round 180 (crisis onset), these were replaced with a mix of normal and disaster messages (8,000 Emotion + 8,000 CrisisMMD v2.0). The detailed labels and text counts are given in the table below. Accuracy was evaluated on the merged test splits of both datasets.

In [ ]:
df = pandas.read_csv("dataset_size.csv")
display(df)

##### page break

### DistilBERT accuracy by clipping norm
tune_layers: 2, learning_rate: 0.001, batch_size: 32, noise_multiplier: 1

We ran 15 rounds on the combined dataset (8,000 Emotion + 8,000 CrisisMMD) with the fixed profile (sigma = 1) and different max_grad_norm values (the DP-SGD clipping norm). Accuracy decreases when the value falls below 0.0001, but it remains unchanged across a wide range from 0.0001 to 1. This contradicts the recommendation of the Google Research paper "How to DP-fy ML: A Practical Guide to Machine Learning with Differential Privacy", which advises choosing the smallest clipping norm (ClipSearch sweep).

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(13, 4))
flwr_accuracy_sub(ax1, "norm/norm10_all", "overall accuracy", 5)
flwr_accuracy_sub(ax2, "norm/norm10_sub", "disaster accuracy", 5, loc="upper left")

##### page break

### Accuracy between privacy profiles
tune_layers: 2, learning_rate: 0.001, batch_size: 32, max_grad_norm: 0.0001

Two unfrozen layers plus the classifier were chosen as a middle ground between reasonable accuracy and a reduced number of parameters exposed to the privacy noise. During normal rounds, all DPA-DP profiles show slightly worse but comparable accuracy to the fixed baseline. After the simulated disaster onset at round 180, all profiles show better accuracy than the fixed baseline, with the high profile performing best.

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(13, 4))
flwr_accuracy_sub(ax1, "profiles/flwr_all", "overall accuracy", 160)
flwr_accuracy_error(ax2, "profiles/flwr_all", 160)

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(13, 4))
flwr_accuracy_sub(ax1, "profiles/flwr_sub", "disaster accuracy", 160)
flwr_accuracy_sub(ax2, "profiles/flwr_all", "overall loss", 160, key="loss", loc="upper right")

##### page break

In [ ]:
accuracy_summary("profiles/flwr_all")

In [ ]:
accuracy_summary("profiles/flwr_sub")